# Lisa: tunnuste lisamine ja eemaldamine

See uuriv laiendus küsib, milline koormuse ajalugu, dünaamika, kalender ja viivitatud ilm pakuvad lisainfot. Kõik tunnused peavad olema prognoosihetkeks teada. Valik toimub 2017–2018; hilisemad 2019 ja 2020 tulemused ei määra tunnusevarianti.

In [ ]:
from pathlib import Path
from functools import partial
import sys
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "run_pipeline.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.common import latest_run, read_json
from src.reporting import show_table, show_figure, show_run
from src.expanded_reporting import display_table

EXPANDED = latest_run("latest_expanded_run.json")
captions = read_json(EXPANDED / "reports/figure_interpretations.json")
expanded_table = partial(show_table, EXPANDED, formatter=display_table)


def expanded_figure(name):
    show_figure(EXPANDED, "figures/" + name, captions[name], width=1100)


show_run(EXPANDED, expanded=True)

In [ ]:
expanded_figure("01_temporal_design.png")

## Tunnuste põhjendused

Nelja nädala sama sihttunni ajalugu kirjeldab korduvat profiili. Ramp ja kiirendus kirjeldavad üleminekuid. Mitme ajaskaala aknad eristavad taset ning hajuvust. Linnade erinevused kontrollivad, kas piirkondlikku teavet läheb keskmistamisel kaduma. Jahutusvajaduse proksid on hüpoteetilised sisendtunnused, mille kasu tuleb mõõta.

In [ ]:
catalog = pd.read_csv(EXPANDED / "processed/feature_catalog.csv")
summary = (
    catalog.groupby("group", sort=False)
    .agg(Tunnuste_arv=("feature", "size"), Põhjendus=("rationale", "first"))
    .reset_index()
)
from src.expanded_reporting import GROUP_NAMES

summary["group"] = summary["group"].map(GROUP_NAMES)
summary = summary.rename(columns={"group": "Tunnuserühm", "Tunnuste_arv": "Tunnuste arv"})
display(Markdown(display_table(summary)))
display(
    Markdown(
        "Tabel seob kaheksa uut rühma kontrollitavate modelleerimishüpoteesidega. Arv näitab rühma tunnuseid; sama või korreleeritud info võib olla olemas ka teistes rühmades. Kõigi tunnuste automaatset kasulikkust ei eeldata."
    )
)

In [ ]:
expanded_table(
    "selection/feature_ranking.csv", ["feature_set", "MAE", "RMSE", "fold_MAE_std", "n_features"]
)
display(
    Markdown(
        "Tabel kirjeldab HistGBR-i kahe valikuaasta kaalutud vigu ühisel maskil. Foldide standardhälve mõõdab ajalist erinevust; see ei ole statistiline usaldusvahemik. Parim lisamisvariant fikseeriti enne hilisemat mudelivõrdlust."
    )
)
expanded_figure("02_feature_additions.png")
expanded_figure("03_feature_ablation.png")

## Mida saab järeldada?

Lisamiskatse hindab rühma mõju baasi suhtes, eemalduskatse kõigi teiste rühmade olemasolul. Erinev tulemus võib tuleneda asendatavast infost ja interaktsioonidest. HistGBR-iga sõelumine võib eelistada selle perekonna vajadusi; seetõttu võrreldakse hiljem igal mudelil ka algset baasi. Ebaõnnestunud variandid jäävad CSV-tulemustesse.